# RUN__pdf_ocr_summary — trạng thái parse báo cáo tài chính, theo cổ phiếu

Đọc **toàn bộ** `raw_data/cafef/financials/statements/**/*.csv` (mọi template: `bank`, `corp`, …)
và tóm tắt thành một bảng, mỗi dòng một cổ phiếu, **6 cột**:

| cột | nghĩa |
|---|---|
| `exchange` | sàn niêm yết, đọc từ chính cột `exchange` của CSV |
| `first_report` | **quý sớm nhất có BÁO CÁO QUÝ** — quý sớm nhất mà cổ phiếu này có ít nhất một statement `source='pdf'` đọc ra từ một *filing quý*; báo cáo năm (`FY-*`) không tính |
| `complete` | **đã thực sự hoàn thành chưa** — `True` khi mỗi statement, tính từ **filing quý đầu tiên của chính nó**, không còn quý nào `missing` |
| `balance_sheet` | **quý muộn nhất bị missing** trong bảng cân đối kế toán |
| `income_statement` | quý muộn nhất bị missing trong báo cáo kết quả kinh doanh |
| `cash_flow` | quý muộn nhất bị missing trong báo cáo lưu chuyển tiền tệ |

Quý hiển thị dạng **`2008-Q4`** — dạng sắp xếp được, và đúng dạng `--quarters` /
`QUARTERS` của `pdf_ocr_job` nhận, nên một ô ở bảng này dán thẳng vào lệnh parse được.
(Trên đĩa CSV vẫn ghi `Q4-2008`; cột `period` gốc được giữ nguyên trong `records`.)

Đọc bảng: sau `first_report` thì doanh nghiệp bắt đầu nộp báo cáo quý; **sau quý ghi trong ba cột
sau thì statement đó liền mạch** — không còn quý nào thiếu. Ô `—` ở ba cột cuối nghĩa là statement đó
**không thiếu quý nào**; ô `—` ở `first_report` nghĩa là **chưa có filing quý nào đọc được** — cùng
một ký hiệu, hai nghĩa khác nhau, nên cell 3 in cảnh báo riêng cho trường hợp thứ hai.

⚠️ **`complete` KHÔNG phải `balance_sheet == income_statement == cash_flow == first_report`.**
Ba cột sau là *quý muộn nhất bị missing*, nên một statement liền mạch khi quý missing cuối cùng của
nó nằm **trước** quý bắt đầu của nó (hoặc là `—`, không thiếu quý nào); một ô **bằng** quý bắt đầu
có nghĩa ngược lại — đúng quý đó bị missing.

⚠️ **Và mốc bắt đầu là của TỪNG statement, không phải `first_report` chung — BID là lý do.**
`first_report = 2011-Q3` vì bảng cân đối và lưu chuyển tiền tệ đọc được từ filing Q3-2011; nhưng
KQKD quý là **lũy kế** và BID không nộp Q1/Q2-2011, nên quý 3 không có gì để trừ — nó **không thể
tạo ra**, và chuỗi KQKD của BID thật sự bắt đầu ở **2012-Q1**. Đo cả ba từ `first_report` chung thì
BID mãi mãi `False` vì đúng một ô không thể parse; đo từ mốc riêng thì BID là ticker **duy nhất**
hiện `True`.

⚠️ **`first_report` chỉ đếm filing QUÝ, và đó là điểm khác so với "quý sớm nhất có số liệu".**
Nhiều cổ phiếu có số liệu từ rất sớm nhưng đều lấy từ **báo cáo năm đã kiểm toán**: BID có
Q4-2008/Q4-2009/Q4-2010 đọc từ `FY-*` nhưng báo cáo quý đầu tiên là **Q3-2011**. Quý càng sớm mà
chỉ có `FY-*` thì chuỗi càng thưa — mỗi năm đúng một điểm.

⚠️ **`missing` là câu trả lời đúng, không phải lỗi.** Theo `CLAUDE.md` §5 rule 24 một số liệu chỉ
được lấy từ PDF gốc; quý nào không có filing hoặc không đọc được thì ghi `missing`. Rất nhiều ô ở
đây là *doanh nghiệp không nộp báo cáo quý đó* (BID/BSR/TCB chỉ nộp báo cáo năm trong các năm đầu),
chứ không phải OCR hỏng.

⚠️ Notebook này **chỉ đọc**, không parse, không OCR, không ghi gì vào `raw_data/`.

## 1 · Định vị thư mục

⚠️ Đường dẫn được dò ngược từ thư mục hiện hành lên tới repo root, nên notebook chạy được cả khi
kernel mở ở `src/kaggle_gpu/` lẫn ở repo root — và **in ra thư mục thật sự đã đọc** (`CWD-1`:
một `STATEMENTS_DIR` tương đối đọc nhầm thư mục rỗng trông y hệt một ticker chưa parse).

In [21]:
from pathlib import Path

import pandas as pd

REPORTS = ["balance_sheet", "income_statement", "cash_flow"]
REL_STATEMENTS = Path("raw_data/cafef/financials/statements")


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / REL_STATEMENTS).is_dir():
            return candidate
    raise FileNotFoundError(f"khong tim thay {REL_STATEMENTS} tu {here} tro len")


REPO_ROOT = _repo_root()
STATEMENTS_DIR = REPO_ROOT / REL_STATEMENTS

print("cwd            :", Path.cwd())
print("repo root      :", REPO_ROOT)
print("statements dir :", STATEMENTS_DIR)

cwd            : d:\GIT\master-thesis\src\kaggle_gpu
repo root      : D:\GIT\master-thesis
statements dir : D:\GIT\master-thesis\raw_data\cafef\financials\statements


## 2 · Đọc mọi CSV thành một bảng dài

Chỉ lấy các cột meta cần thiết. `encoding='utf-8-sig'` là bắt buộc — cột đầu tiên của các file này
mang BOM, nếu không sẽ thành `﻿symbol`.

In [22]:
META = ["symbol", "exchange", "template", "period", "year", "quarter", "source", "document"]

frames = []
for path in sorted(STATEMENTS_DIR.glob("*/*/*.csv")):
    report = path.parent.name
    if report not in REPORTS:
        print(f"WARNING: bo qua {path} — thu muc bao cao la {report!r}")
        continue
    part = pd.read_csv(path, usecols=META, encoding="utf-8-sig")
    part["report"] = report
    part["file"] = path.relative_to(REPO_ROOT).as_posix()
    frames.append(part)

if not frames:
    raise FileNotFoundError(f"khong co file .csv nao trong {STATEMENTS_DIR}")

records = pd.concat(frames, ignore_index=True)
records["year"] = records["year"].astype(int)
records["quarter"] = records["quarter"].astype(int)
records["source"] = records["source"].fillna("missing")

# mot symbol niem yet tren hai san se pha khoa ticker — doi khoa thay vi im lang gop nham
pairs = records[["exchange", "symbol"]].drop_duplicates()
if pairs["symbol"].duplicated().any():
    records["ticker"] = records["exchange"] + "_" + records["symbol"]
else:
    records["ticker"] = records["symbol"]

# quy -> mot so nguyen tang dan, de lay min/max ma khong sap xep chuoi 'Q4-2008'
records["rank"] = records["year"] * 4 + records["quarter"]

# dang hien thi YYYY-QQ: sap xep duoc, va la dang --quarters cua pdf_ocr_job
records["quarter_id"] = records["year"].astype(str) + "-Q" + records["quarter"].astype(str)

# BAO CAO QUY vs BAO CAO NAM — doc tu TEN FILE, la bang chung truc tiep:
# `CafeFPdfScraper._base_name` dat tien to `Q{1..4}-{nam}` cho mot filing quy va `FY-{nam}`
# cho bao cao nam (CafeF xep bao cao nam vao quarter 5).
# ⚠️ `assurance` KHONG phan biet duoc: Q2 la `reviewed`, Q1/Q3 `unaudited`, FY `audited` —
# doc theo assurance thi mot filing quy da soat xet se bi doi thanh bao cao nam.
# ⚠️ Va `documents()` UU TIEN bao cao nam da kiem toan cho Q4, nen mot dong Q4 hau nhu luon
# mang `FY-*` du doanh nghiep co nop bao cao quy 4 — vi vay `first_report` gan nhu luon roi
# vao Q1..Q3, va do la dung.
QUARTERLY_DOCUMENT = r"^Q[1-4]-\d{4}_"
records["quarterly_filing"] = (
    records["document"].fillna("").astype(str).str.match(QUARTERLY_DOCUMENT)
)

unexpected = sorted(set(records["source"]) - {"pdf", "missing"})
if unexpected:
    print(f"WARNING: gia tri source ngoai du kien: {unexpected} — bang duoi coi chung KHONG phai 'co bao cao'")

print(f"{len(frames)} file / {records['ticker'].nunique()} co phieu / {len(records)} dong")
print(records["source"].value_counts().to_dict())
records

21 file / 7 co phieu / 1182 dong
{'pdf': 1002, 'missing': 180}


,symbol,exchange,template,period,year,quarter,source,document,report,file,ticker,rank,quarter_id,quarterly_filing
0,ACB,HOSE,bank,Q1-2008,2008,1,missing,NaN,balance_sheet,raw_data/cafef/financials/statements/bank/bala...,ACB,8033,2008-Q1,False
1,ACB,HOSE,bank,Q2-2008,2008,2,missing,NaN,balance_sheet,raw_data/cafef/financials/statements/bank/bala...,ACB,8034,2008-Q2,False
2,ACB,HOSE,bank,Q3-2008,2008,3,missing,NaN,balance_sheet,raw_data/cafef/financials/statements/bank/bala...,ACB,8035,2008-Q3,False
3,ACB,HOSE,bank,Q4-2008,2008,4,missing,NaN,balance_sheet,raw_data/cafef/financials/statements/bank/bala...,ACB,8036,2008-Q4,False
4,ACB,HOSE,bank,Q1-2009,2009,1,missing,NaN,balance_sheet,raw_data/cafef/financials/statements/bank/bala...,ACB,8037,2009-Q1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1177,VIC,HOSE,corp,Q4-2013,2013,4,missing,NaN,income_statement,raw_data/cafef/financials/statements/corp/inco...,VIC,8056,2013-Q4,False
1178,VIC,HOSE,corp,Q1-2014,2014,1,pdf,Q1-2014_bao_cao_tai_chinh_hop_nhat_quy_1_nam_2...,income_statement,raw_data/cafef/financials/statements/corp/inco...,VIC,8057,2014-Q1,True
1179,VIC,HOSE,corp,Q2-2014,2014,2,pdf,Q2-2014_bao_cao_tai_chinh_hop_nhat_quy_2_nam_2...,income_statement,raw_data/cafef/financials/statements/corp/inco...,VIC,8058,2014-Q2,True
1180,VIC,HOSE,corp,Q3-2014,2014,3,pdf,Q3-2014_bao_cao_tai_chinh_hop_nhat_quy_3_nam_2...,income_statement,raw_data/cafef/financials/statements/corp/inco...,VIC,8059,2014-Q3,True


## 3 · Bảng tóm tắt — 6 cột

`exchange` là sàn của cổ phiếu; `first_report` lấy trên **cả ba** statement — quý sớm nhất có bất
kỳ statement nào đọc được **từ một filing quý**; ba cột cuối tính riêng từng statement.

Thứ tự ưu tiên giữ nguyên như cũ (`source='pdf'`, gộp cả ba statement, lấy `rank` nhỏ nhất);
chỉ thêm đúng một điều kiện lọc: dòng đó phải đến từ báo cáo quý, không phải báo cáo năm.

`complete` là cột thứ 3, kiểu `bool` (không phải chuỗi, để lọc thẳng bằng `summary[summary["complete"]]`):
**mỗi statement, tính từ filing quý đầu tiên của CHÍNH NÓ, không còn quý `missing` nào.**

⚠️ **Mốc bắt đầu là của từng statement, không phải `first_report` chung.** BID: `first_report` là
2011-Q3 (bảng cân đối + lưu chuyển tiền tệ đọc từ filing Q3-2011), nhưng KQKD quý 3 là lũy kế và
BID không nộp Q1/Q2-2011 nên không có gì để trừ — chuỗi KQKD của nó bắt đầu ở 2012-Q1. Ba mốc thật
của BID là `2011-Q3 / 2012-Q1 / 2011-Q3`, và mọi quý missing đều nằm trước mốc của mình.

⚠️ **Mẫu số không phải toàn bộ lịch sử.** Quý missing *trước* mốc bắt đầu là câu trả lời đúng —
doanh nghiệp chưa nộp báo cáo quý (BID chỉ nộp báo cáo năm cho 2008/2009/2010), nên đưa chúng vào
mẫu số là đo **lịch nộp báo cáo** chứ không phải đo parser. Đây đúng là phép re-scope `CLAUDE.md`
§6-2-quindecies đã làm cho BID: bỏ lịch nộp ra khỏi mẫu số thì 80.0 % thành 94.2 %.

⚠️ **Giá phải trả, và nói ra thay vì giấu: khoảng trống nằm TRƯỚC mốc của statement đó được bỏ
qua.** Ở đây không tách được *"không ai nộp / không thể tạo ra"* khỏi *"có filing mà parse hỏng"*,
vì dòng `missing` **không mang `document`** — §6-2-terdecies đã xoá provenance khỏi dòng missing,
nên notebook này không còn bằng chứng nào để phân biệt. Tách được thì phải đọc chỉ mục PDF.

⚠️ **`complete = False` nghĩa là *chưa chứng minh được liền mạch*, không phải *parser hỏng*.**
Một quý missing sau mốc vẫn có thể là quý doanh nghiệp không nộp (BSR không nộp báo cáo quý nào
trước H2-2018); phân biệt cũng cần chính chỉ mục PDF (`raw_data/cafef/pdfs/index/`) — việc mà
`_decumulate` và `pdf_ocr_merge` phải làm, còn ở đây thì không.

⚠️ Statement nào chưa có filing quý nào đọc được thì cả ticker là `False`: không biết chuỗi bắt đầu
từ đâu thì không thể nói nó liền mạch.


In [23]:
# khoa ticker o cell 2 da bao dam moi ticker chi thuoc mot san
exchange = records.groupby("ticker")["exchange"].first()

parsed = records[records["source"] == "pdf"]

# ⚠️ CHI TINH TREN FILING QUY. Uu tien van y het cu — quy som nhat, tren ca ba statement,
# chi lay dong `source='pdf'` — chi them dieu kien `quarterly_filing` (xem cell 2).
quarterly = parsed[parsed["quarterly_filing"]]
first_report = (
    quarterly.loc[quarterly.groupby("ticker")["rank"].idxmin()]
    .set_index("ticker")["quarter_id"]
    .rename("first_report")
)

# mot ticker co so lieu nhung chua co filing quy nao doc duoc se ra `—`, trung ky hieu voi
# "khong thieu quy nao" o ba cot sau — noi ra thay vi de nguoi doc tu doan
only_annual = sorted(set(parsed["ticker"]) - set(quarterly["ticker"]))
if only_annual:
    print("WARNING: chi co bao cao nam, chua co filing quy nao doc duoc:",
          ", ".join(only_annual))

missing = records[records["source"] == "missing"]
last_missing = (
    missing.loc[missing.groupby(["ticker", "report"])["rank"].idxmax()]
    .pivot(index="ticker", columns="report", values="quarter_id")
    .reindex(columns=REPORTS)
)

index = pd.Index(sorted(records["ticker"].unique()), name="ticker")

# ⚠️ `complete` DO TREN TUNG STATEMENT, moi statement tinh tu FILING QUY DAU TIEN CUA CHINH NO —
# khong phai tu `first_report` chung. BID la ly do: `first_report` = 2011-Q3 vi bang can doi va luu
# chuyen tien te doc duoc tu filing Q3-2011, nhung KQKD quy la LUY KE va BID khong nop Q1/Q2-2011
# nen quy 3 khong co gi de tru — o do KHONG THE tao ra, va chuoi KQKD cua BID bat dau o 2012-Q1.
# Do ca ba tu `first_report` chung thi BID mai mai `False` vi dung mot o khong the parse.
# ⚠️ Khong phai `cot == first_report`: ba cot tren la QUY MUON NHAT BI MISSING, nen mot statement
# lien mach khi quy missing cuoi cung cua no nam TRUOC moc bat dau cua no, con mot o BANG moc do
# nghia la dung quy do bi missing.
# ⚠️ Gia phai tra: khoang trong nam TRUOC moc cua statement do duoc BO QUA, va o day khong tach
# duoc "khong ai nop / khong the tao ra" khoi "co filing ma parse hong" — dong `missing` khong mang
# `document` (§6-2-terdecies da xoa provenance khoi dong missing), tach duoc thi phai doc chi muc PDF.
first_rank = quarterly.groupby(["ticker", "report"])["rank"].min()
first_rank_wide = first_rank.unstack("report").reindex(index=index, columns=REPORTS)
last_missing_rank = (
    missing.groupby(["ticker", "report"])["rank"]
    .max()
    .unstack("report")
    .reindex(index=index, columns=REPORTS)
)
# so sanh voi NaN tra ve False, nen mot statement KHONG THIEU QUY NAO tinh la lien mach; nhung mot
# statement chua co filing quy nao doc duoc cung tra ve False, phai chan rieng bang `notna()`
blocking = last_missing_rank.ge(first_rank_wide)
complete = (~blocking.any(axis=1) & first_rank_wide.notna().all(axis=1)).rename("complete")

summary = (
    pd.DataFrame(index=index)
    .join(exchange)
    .join(first_report)
    .join(last_missing)
    .fillna("—")
    .join(complete)  # sau fillna, de `complete` giu kieu bool thay vi thanh chuoi
    [["exchange", "first_report", "complete", *REPORTS]]  # `complete` la cot thu 3
)

# mau so cua cot `complete`: con bao nhieu quy missing tu moc bat dau cua statement do tro di
still = (
    missing.join(first_rank.rename("first_rank"), on=["ticker", "report"])
    .loc[lambda d: d["rank"] >= d["first_rank"]]
    .groupby("ticker")
    .size()
    .sort_values(ascending=False)
)
print(f"complete: {int(complete.sum())}/{len(index)} co phieu lien mach tu quy bat dau cua no")
if not still.empty:
    print("con thieu (quy >= moc bat dau cua statement do):",
          ", ".join(f"{t}={n}" for t, n in still.items()))

summary

complete: 1/7 co phieu lien mach tu quy bat dau cua no
con thieu (quy >= moc bat dau cua statement do): CTG=48, VIC=14, BSR=13, TCB=7, ACB=2, VCB=1


,exchange,first_report,balance_sheet,income_statement,cash_flow,complete
ticker,,,,,,
ACB,HOSE,2009-Q2,2009-Q3,2009-Q4,2009-Q3,False
BID,HOSE,2011-Q3,2011-Q2,2011-Q3,2011-Q2,True
BSR,HOSE,2017-Q3,2019-Q4,2020-Q4,2018-Q2,False
CTG,HOSE,2009-Q1,2019-Q1,2025-Q4,2024-Q1,False
TCB,HOSE,2012-Q2,2013-Q1,2012-Q3,2021-Q1,False
VCB,HOSE,2009-Q1,2009-Q2,2008-Q4,2009-Q2,False
VIC,HOSE,2008-Q2,2010-Q4,2013-Q4,2014-Q1,False


## 4 · Phụ — đếm quý đã parse / còn thiếu

Không nằm trong 4 cột được hỏi, nhưng là mẫu số để đọc bảng trên: một `first_report` sớm mà
`coverage` thấp nghĩa là chuỗi dài nhưng rỗng, chứ không phải lịch sử dài.

⚠️ `quarters` là số dòng CSV, tức khoảng thời gian file phủ — **không** phải số quý doanh
nghiệp thực sự nộp báo cáo. Quý không có filing nào cũng nằm trong mẫu số này.

In [24]:
tally = (
    records.groupby(["ticker", "report"])["source"]
    .value_counts()
    .unstack("source")
    .reindex(columns=["pdf", "missing"])
    .fillna(0)
    .astype(int)
)
tally["quarters"] = tally["pdf"] + tally["missing"]
tally["coverage"] = (tally["pdf"] / tally["quarters"]).round(3)
tally

source                   pdf  missing  quarters  coverage
ticker report                                            
ACB    balance_sheet      67        6        73     0.918
       cash_flow          66        7        73     0.904
       income_statement   66        7        73     0.904
BID    balance_sheet      62        8        70     0.886
       cash_flow          61        9        70     0.871
       income_statement   58       12        70     0.829
BSR    balance_sheet      10        7        17     0.588
       cash_flow          13        4        17     0.765
       income_statement    6       11        17     0.353
CTG    balance_sheet      61        9        70     0.871
       cash_flow          61        9        70     0.871
       income_statement   35       35        70     0.500
TCB    balance_sheet      57       10        67     0.851
       cash_flow          52       15        67     0.776
       income_statement   59        8        67     0.881
VCB    balance_sheet      68        2        70     0.971
       cash_flow          69        1        70     0.986
       income_statement   69        1        70     0.986
VIC    balance_sheet      20        7        27     0.741
       cash_flow          20        7        27     0.741
       income_statement   22        5        27     0.815